# Financial Rag Audit Engine

**Project:** Financial Planning & Analysis Intelligence Platform

**Notebook:** `09-financial-rag-audit-engine.ipynb`

In [1]:
# ==========================================
# Notebook 09
# Financial RAG Audit Engine
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
financial_df = pd.read_csv("../data/financial_macro_dataset.csv")

financial_df.head()

,ticker,quarter,revenue_million,gross_margin_pct,operating_income_million,net_income_million,eps,earnings_call,risk_factors,mda_section,...,confidence,weighted_sentiment,guidance_score,positive_guidance_score,total_risk_score,normalized_risk_score,financial_document,best_macro_event,similarity_score,macro_risk_score
0,ABC,2023-Q1,120,58,22,16,1.20,\n Demand remained strong across enterp...,\n Inflation remains a concern.\n ...,\n Management believes demand trends re...,...,0.956167,0.956167,1,1,2,1.0,\n Demand remained strong across enterp...,High Inflation Environment,0.583904,0.583904
1,ABC,2023-Q2,128,60,25,18,1.35,\n Enterprise adoption accelerated.\n ...,\n Competitive pressure exists.\n ...,\n New product launches contributed pos...,...,0.957692,0.957692,1,2,1,0.5,\n Enterprise adoption accelerated.\n ...,Currency Volatility Event,0.624035,0.624035
2,ABC,2023-Q3,138,61,28,21,1.52,\n Customer demand exceeded expectation...,\n Geopolitical uncertainty remains.\n ...,\n Strong customer growth across region...,...,0.951876,0.951876,2,0,1,0.5,\n Customer demand exceeded expectation...,Regulatory Pressure Cycle,0.565232,0.565232
3,ABC,2023-Q4,150,63,32,24,1.72,\n Record quarter performance.\n ...,\n Macroeconomic slowdown remains possi...,\n Revenue growth exceeded internal exp...,...,0.956653,0.956653,1,1,0,0.0,\n Record quarter performance.\n ...,Interest Rate Shock,0.499716,0.499716
4,ABC,2024-Q1,158,62,34,26,1.85,\n Strong start to the fiscal year driv...,\n Increased cyber security threats req...,\n Gross margin slightly contracted due...,...,0.922371,0.922371,1,0,0,0.0,\n Strong start to the fiscal year driv...,COVID Supply Chain Crisis,0.381064,0.381064


In [3]:
print("Rows:", len(financial_df))

print("Columns:", len(financial_df.columns))

financial_df.columns

Rows: 12
Columns: 26


Index(['ticker', 'quarter', 'revenue_million', 'gross_margin_pct',
       'operating_income_million', 'net_income_million', 'eps',
       'earnings_call', 'risk_factors', 'mda_section', 'combined_text',
       'clean_text', 'character_count', 'sentences', 'chunks', 'sentiment',
       'confidence', 'weighted_sentiment', 'guidance_score',
       'positive_guidance_score', 'total_risk_score', 'normalized_risk_score',
       'financial_document', 'best_macro_event', 'similarity_score',
       'macro_risk_score'],
      dtype='object')

In [4]:
financial_df["rag_document"] = (
    financial_df["earnings_call"].fillna("")
    + " "
    + financial_df["risk_factors"].fillna("")
    + " "
    + financial_df["mda_section"].fillna("")
)

In [5]:
financial_df[["quarter", "rag_document"]]

,quarter,rag_document
0,2023-Q1,\n Demand remained strong across enterp...
1,2023-Q2,\n Enterprise adoption accelerated.\n ...
2,2023-Q3,\n Customer demand exceeded expectation...
3,2023-Q4,\n Record quarter performance.\n ...
4,2024-Q1,\n Strong start to the fiscal year driv...
5,2024-Q2,\n New AI-driven product modules saw re...
6,2024-Q3,\n European expansion is yielding solid...
7,2024-Q4,"\n An exceptional finish to 2024, cross..."
8,2025-Q1,\n We carried the strong closing moment...
9,2025-Q2,\n Our SaaS migration is officially com...


In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [7]:
document_embeddings = embedding_model.encode(
    financial_df["rag_document"].tolist(), show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
document_embeddings.shape

(12, 768)

In [9]:
embedding_dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

In [10]:
index.add(np.array(document_embeddings).astype("float32"))

In [11]:
print("Documents Indexed:", index.ntotal)

Documents Indexed: 12


In [12]:
def financial_retrieval(query, top_k=3):

    query_embedding = embedding_model.encode(query)

    distances, indices = index.search(
        np.array([query_embedding]).astype("float32"), top_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        results.append(
            {
                "rank": rank + 1,
                "quarter": financial_df.iloc[idx]["quarter"],
                "distance": float(distances[0][rank]),
                "document": financial_df.iloc[idx]["rag_document"],
            }
        )

    return pd.DataFrame(results)

In [13]:
financial_retrieval("strong customer demand")

,rank,quarter,distance,document
0,1,2023-Q1,0.549402,\n Demand remained strong across enterp...
1,2,2023-Q3,0.830314,\n Customer demand exceeded expectation...
2,3,2023-Q4,0.954851,\n Record quarter performance.\n ...


In [14]:
financial_retrieval("supply chain challenges")

,rank,quarter,distance,document
0,1,2023-Q3,0.940261,\n Customer demand exceeded expectation...
1,2,2023-Q1,1.051198,\n Demand remained strong across enterp...
2,3,2024-Q3,1.304337,\n European expansion is yielding solid...


In [15]:
query = """

Why did revenue forecast increase?

"""

In [16]:
audit_results = financial_retrieval(query, top_k=3)

audit_results

,rank,quarter,distance,document
0,1,2023-Q4,0.971011,\n Record quarter performance.\n ...
1,2,2023-Q1,1.021231,\n Demand remained strong across enterp...
2,3,2024-Q1,1.023360,\n Strong start to the fiscal year driv...


In [17]:
financial_retrieval("revenue growth drivers")

,rank,quarter,distance,document
0,1,2023-Q1,0.943952,\n Demand remained strong across enterp...
1,2,2023-Q4,0.955568,\n Record quarter performance.\n ...
2,3,2023-Q2,0.973696,\n Enterprise adoption accelerated.\n ...


In [18]:
financial_retrieval("margin expansion and profitability")

,rank,quarter,distance,document
0,1,2024-Q3,0.993964,\n European expansion is yielding solid...
1,2,2023-Q3,1.037279,\n Customer demand exceeded expectation...
2,3,2023-Q4,1.086257,\n Record quarter performance.\n ...


In [19]:
financial_retrieval("major business risks")

,rank,quarter,distance,document
0,1,2024-Q3,1.084860,\n European expansion is yielding solid...
1,2,2023-Q3,1.172600,\n Customer demand exceeded expectation...
2,3,2024-Q4,1.175659,"\n An exceptional finish to 2024, cross..."


In [20]:
financial_retrieval("future growth expectations")

,rank,quarter,distance,document
0,1,2023-Q1,1.011263,\n Demand remained strong across enterp...
1,2,2023-Q4,1.056872,\n Record quarter performance.\n ...
2,3,2023-Q3,1.131008,\n Customer demand exceeded expectation...


In [21]:
results = financial_retrieval("enterprise demand growth")

results

,rank,quarter,distance,document
0,1,2023-Q1,0.814697,\n Demand remained strong across enterp...
1,2,2023-Q2,0.910854,\n Enterprise adoption accelerated.\n ...
2,3,2024-Q3,1.108670,\n European expansion is yielding solid...


In [22]:
for doc in results["document"]:

    print("=" * 80)

    print(doc)


        Demand remained strong across enterprise customers.
        We expect continued growth next quarter.
        Supply chain constraints are gradually easing.
         
        Inflation remains a concern.
        Global logistics costs remain elevated.
         
        Management believes demand trends remain healthy.
        Customer retention continues to improve.
        

        Enterprise adoption accelerated.
        Pricing pressure has moderated.
        We expect strong momentum in Q3.
         
        Competitive pressure exists.
        Currency fluctuations may impact revenue.
         
        New product launches contributed positively.
        Pipeline remains healthy.
        

        European expansion is yielding solid initial returns despite local economic headwinds.
        Customer churn hit an all-time low this quarter.
        Strategic partnerships are expanding our addressable market.
         
        Evolving global data privacy laws could increase

In [23]:
audit_queries = [
    "revenue growth",
    "supply chain risk",
    "inflation pressure",
    "future guidance",
    "profitability improvement",
]

In [24]:
audit_logs = []

for query in audit_queries:

    results = financial_retrieval(query)

    for _, row in results.iterrows():

        audit_logs.append(
            {"query": query, "quarter": row["quarter"], "distance": row["distance"]}
        )

In [25]:
audit_logs_df = pd.DataFrame(audit_logs)

audit_logs_df.head()

,query,quarter,distance
0,revenue growth,2023-Q4,0.913985
1,revenue growth,2023-Q1,0.970893
2,revenue growth,2023-Q2,1.003676
3,supply chain risk,2023-Q3,1.157016
4,supply chain risk,2023-Q1,1.261866


In [26]:
faiss.write_index(index, "../data/financial_rag_index.faiss")

In [27]:
audit_logs_df.to_csv("../data/financial_audit_logs.csv", index=False)

In [28]:
financial_df.to_csv("../data/financial_rag_corpus.csv", index=False)

In [29]:
loaded_index = faiss.read_index("../data/financial_rag_index.faiss")

loaded_index.ntotal

12